# 11: O Consolidador (relatório final)

Toda análise termina no Consolidador. Antes dele, a resposta final era só a última mensagem do grafo:
às vezes o texto do setor, às vezes o diagnóstico, nunca tudo junto.

O Consolidador **não é um agente com tools**: é uma única chamada ao LLM com saída estruturada
(`RelatorioFinal`), seguida de ajustes em código e da formatação em markdown.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Uma análise completa da VALE3

Esta célula roda o pipeline inteiro (cerca de 20 a 30 segundos). Como a VALE3 apresenta sinais de anomalia,
o Supervisor despacha o Analyzer e o Investigator em paralelo na Etapa 2. O pipeline completa em apenas 3 voltas
do Supervisor (`loop_count = 3`, contra 5 da versão sequencial anterior).

In [2]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro import app

resultado = app.invoke({"messages": [HumanMessage("Analise a ação VALE3")]})
quem_falou = [m.name for m in resultado["messages"] if m.name]
print("Mensagens no histórico:", quem_falou)
print("loop_count:", resultado["loop_count"])

2026-11-08 23:58:52,749 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-08 23:58:53,997 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, histórico de preços, balanço e indicadores fundamentalistas para VALE3.'


2026-11-08 23:58:53,998 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-08 23:58:54,000 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


2026-11-08 23:58:54,820 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=VALE3.SA


2026-11-08 23:58:54,824 - fluxo_financeiro - INFO - [tool] calcular_metricas_risco: ticker=VALE3.SA


... (34 outputs left out)

2026-11-08 23:59:15,377 - fluxo_financeiro - DEBUG - [router] FINISH → Consolidador


2026-11-08 23:59:15,378 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (7 msgs)


2026-11-08 23:59:17,727 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (3236 chars)


Mensagens no histórico: ['Supervisor', 'YFinance', 'Supervisor', 'Supervisor', 'Analyzer', 'Investigator', 'Consolidador']
loop_count: 3


## 2. O que o Consolidador recebe e devolve

Ele recebe só o pedido do usuário e as respostas de YFinance, Analyzer e Investigator (as tarefas do
Supervisor ficam de fora), cada uma dentro de `<resposta_agente>`. Devolve um `RelatorioFinal`:

In [3]:
from fluxo_financeiro.agents.consolidator import montar_insumos

print("\n\n".join(montar_insumos(resultado["messages"]))[:1500], "...")

<pedido_usuario>Analise a ação VALE3</pedido_usuario>

<resposta_agente nome="YFinance" n="1">
Dados para VALE3.SA:
Empresa: Vale S.A.
Preço Atual: R$ 64.04 (no dia: 2.19%)
Faixa de 52 semanas: R$ 59.42 a R$ 91.62
Valor de mercado: R$ 248.5 bi
P/L:  77.2x
ROE:  0.3%
DY:   6.82%
Setor: Materiais Básicos | Subsetor: Mineração de Minerais Metálicos
Maiores do setor na Brapi (Non-Energy Minerals, por valor de mercado): VALE3 é a 1ª de 12.
  1. VALE3 VALE S.A.: R$ 248.5 bi
  2. GGBR4 GERDAU S.A.: R$ 46.0 bi
  3. CMIN3 CSN MINERAÇÃO S.A.: R$ 28.3 bi
  4. KLBN4 KLABIN S.A.: R$ 21.7 bi
  5. GOAU4 METALURGICA GERDAU S.A.: R$ 14.0 bi
Sumario de Negócio:
A Vale S.A. (VALE3) é uma companhia brasileira de mineração e metais, com foco principal na produção de minério de ferro e de metais para transição energética. A empresa também atua na produção de pelotas de minério de ferro e, no segmento de metais básicos, em produtos como níquel, cobre, cobalto e metais do grupo da platina, além de subprodutos

In [4]:
relatorio = resultado["relatorio"]
for campo in ("resumo", "indicadores", "contexto", "riscos", "limitacoes"):
    print(f"--- {campo}:\n{getattr(relatorio, campo)}\n")

--- resumo:
A ação VALE3 cotada a R$ 64,04 reflete desafios operacionais e financeiros, impulsionados pela queda de 53% no lucro líquido do terceiro trimestre de 2026 para US$ 1,25 bilhão. Essa retração foi causada principalmente pela redução de 10% no preço médio realizado dos finos de minério de ferro, provisões da Samarco e aumento de custos, apesar da sólida geração de caixa operacional.

--- indicadores:
['Preço Atual: R$ 64.04', 'Faixa de 52 semanas: R$ 59.42 a R$ 91.62', 'Valor de mercado: R$ 248.5 bi', 'P/L: 77.2x', 'ROE: 0.3%', 'DY: 6.82%', 'Sharpe Ratio: -0.11', 'Sortino Ratio: -0.18', 'Max Drawdown: -31.5%', 'VaR 95% (diário): -2.91%', 'Beta (vs IBOV): 0.39', 'Alpha (anual): -3.4%', 'Piotroski F-Score: 5/9', 'SELIC (meta): 13.75% a.a.', 'IPCA acumulado em 12 meses: 4.58%']

--- contexto:
A Vale atua no setor de Materiais Básicos como a maior empresa entre os doze principais nomes do segmento na B3. O modelo de negócios é pautado na integração logística e em dois grandes negó

## 3. O diagnóstico é copiado, não resumido

O LLM é instruído a deixar `diagnostico` vazio; o código copia o texto do Investigator literalmente. Assim as
quatro seções (Causa, Natureza, Evidências, Impacto) nunca são reescritas.

In [5]:
texto_investigator = next((m.text for m in resultado["messages"] if m.name == "Investigator"), None)
print("Investigator rodou:", texto_investigator is not None)
print("Diagnóstico idêntico ao texto do Investigator:", relatorio.diagnostico == texto_investigator)

Investigator rodou: True
Diagnóstico idêntico ao texto do Investigator: True


## 4. A mesma função, chamada à parte

`consolidar(state)` é uma função comum: dá para chamá-la com qualquer state. Uma chamada ao LLM.

In [6]:
from fluxo_financeiro.agents.consolidator import consolidar, formatar_relatorio, relatorio_de_emergencia

de_novo = consolidar(resultado)
print("Indicadores na 2a chamada:", len(de_novo.indicadores), "| na análise:", len(relatorio.indicadores))

Indicadores na 2a chamada: 15 | na análise: 15


## 5. Do relatório estruturado ao markdown

`formatar_relatorio` monta o markdown em código. Seções vazias são omitidas.

In [7]:
from IPython.display import Markdown

Markdown(formatar_relatorio(relatorio, "VALE3"))

# Relatório: VALE3

## Resumo
A ação VALE3 cotada a R$ 64,04 reflete desafios operacionais e financeiros, impulsionados pela queda de 53% no lucro líquido do terceiro trimestre de 2026 para US$ 1,25 bilhão. Essa retração foi causada principalmente pela redução de 10% no preço médio realizado dos finos de minério de ferro, provisões da Samarco e aumento de custos, apesar da sólida geração de caixa operacional.

## Indicadores
- Preço Atual: R$ 64.04
- Faixa de 52 semanas: R$ 59.42 a R$ 91.62
- Valor de mercado: R$ 248.5 bi
- P/L: 77.2x
- ROE: 0.3%
- DY: 6.82%
- Sharpe Ratio: -0.11
- Sortino Ratio: -0.18
- Max Drawdown: -31.5%
- VaR 95% (diário): -2.91%
- Beta (vs IBOV): 0.39
- Alpha (anual): -3.4%
- Piotroski F-Score: 5/9
- SELIC (meta): 13.75% a.a.
- IPCA acumulado em 12 meses: 4.58%

## Notícias e setor
A Vale atua no setor de Materiais Básicos como a maior empresa entre os doze principais nomes do segmento na B3. O modelo de negócios é pautado na integração logística e em dois grandes negócios: Iron Solutions e Energy Transition Metals. O cenário macroeconômico global apresenta preços do minério de ferro pressionados e demanda cautelosa da China, enquanto o setor mineral brasileiro projeta investimentos de US$ 76,9 bilhões entre 2026 e 2030. A administração avalia calibrar a produção para 2027 com um corte de cerca de 5 milhões de toneladas.

## Diagnóstico da anomalia
- Causa identificada: A queda de 53% no lucro líquido do 3T26 foi causada pela redução de 10% no preço médio realizado dos finos de minério de ferro, por provisões extraordinárias e revisões relacionadas à Samarco, além do aumento dos custos caixa e pressões cambiais.
- Natureza do evento: cíclico
- Evidências: Lucro líquido caiu para US$ 1,25 bi no 3T26 ante US$ 2,69 bi no 3T25.; Preço médio realizado dos finos de minério de ferro caiu 10% a/a para US$ 85,10 por tonelada.; Ajuste e revisão na provisão relacionada à Samarco com impacto negativo estimado em cerca de US$ 322 milhões.; Custo caixa C1 subiu 12% a/a para US$ 23,10/t.; Produção de minério de ferro recuou 6% na base anual, totalizando 88,4 milhões de toneladas.
- Impacto no investimento: Demonstra que a rentabilidade da companhia permanece altamente sensível à oscilação dos preços internacionais do minério de ferro e ao cenário macroeconômico de custos e câmbio, exigindo monitoramento da volatilidade das commodities.
Fontes: bpmoney.com.br, infomoney.com.br, vale.com, advfn.com, revistamineracao.com.br, brazilstockguide.com, moneytimes.com.br, abril.com.br, euqueroinvestir.com, estadao.com.br

## Riscos
- Lucro líquido do trimestre encerrado em 2026-09-30 caiu 53% em relação ao mesmo trimestre do ano anterior (US$ 1.25 bi contra US$ 2.69 bi).
- Sharpe Ratio de -0.11 considerado fraco.
- Alpha anual de -3.4%, ficando abaixo do esperado pelo CAPM.
- Aumento de 21.2% na dívida corrente.
- Queda na Gross Margin para 35.0% vs 36.2%.
- Redução de 10% no preço médio realizado dos finos de minério de ferro.
- Ajuste e revisão na provisão relacionada à Samarco com impacto negativo estimado em cerca de US$ 322 milhões.
- Custo caixa C1 subiu 12% a/a para US$ 23,10/t.
- Produção de minério de ferro recuou 6% na base anual, totalizando 88,4 milhões de toneladas.


## 6. Se o LLM do Consolidador falhar

A análise já custou várias chamadas; perder tudo no último passo seria ruim. Se `consolidar` levantar erro
(cota, rede, saída inválida), o nó usa `relatorio_de_emergencia`: as respostas dos agentes juntadas, sem LLM,
e `relatorio = None` no estado.

In [8]:
print(relatorio_de_emergencia(resultado["messages"])[:800], "...")

> Não foi possível consolidar o relatório; seguem as respostas dos agentes.

## YFinance
Dados para VALE3.SA:
Empresa: Vale S.A.
Preço Atual: R$ 64.04 (no dia: 2.19%)
Faixa de 52 semanas: R$ 59.42 a R$ 91.62
Valor de mercado: R$ 248.5 bi
P/L:  77.2x
ROE:  0.3%
DY:   6.82%
Setor: Materiais Básicos | Subsetor: Mineração de Minerais Metálicos
Maiores do setor na Brapi (Non-Energy Minerals, por valor de mercado): VALE3 é a 1ª de 12.
  1. VALE3 VALE S.A.: R$ 248.5 bi
  2. GGBR4 GERDAU S.A.: R$ 46.0 bi
  3. CMIN3 CSN MINERAÇÃO S.A.: R$ 28.3 bi
  4. KLBN4 KLABIN S.A.: R$ 21.7 bi
  5. GOAU4 METALURGICA GERDAU S.A.: R$ 14.0 bi
Sumario de Negócio:
A Vale S.A. (VALE3) é uma companhia brasileira de mineração e metais, com foco principal na produção de minério de ferro e de metais para transição energé ...


## Sem recomendação de compra ou venda

O relatório descreve o que os dados mostram e não dá veredito nem preço-alvo. Um "compre" escrito por um LLM a
partir de poucas fontes passaria uma confiança que os dados não sustentam.

## O que observar

- A análise da VALE3 rodou com `loop_count = 3`: o Supervisor despachou o Analyzer e o Investigator em paralelo na Etapa 2 via `Send`, reduzindo o tempo total de resposta sem perder nenhuma seção no relatório final.
- `limitacoes`: lista o que faltou ou falhou (busca sem resultado, etapa que não rodou). Parte delas é escrita em código, não depende de o LLM perceber.
- Os números de `indicadores` precisam aparecer nas respostas dos agentes: o prompt proíbe inventar ou recalcular.
- O diagnóstico do Investigator entra intacto no relatório final, preservando a análise causal da anomalia.
- A segunda chamada a `consolidar` pode escrever frases diferentes da primeira (o LLM não é determinístico), mas os números são os mesmos, porque vêm dos insumos.
- O relatório de emergência é feio, mas não perde nada do que os agentes produziram se o LLM do Consolidador falhar.